# Before the Signal Dies: relative outage susceptibility index

This notebook ranks every mapped Western Australian locality by **relative mobile-network outage susceptibility** and assigns a **Low / Medium / High** band.

**How it works**

1. Seven measured features are scaled to 0–1, where 1 is the worst value in WA.
2. They are averaged into two parts:
   - **Hazard exposure**: bushfire-prone area, mapped floodplain, recorded bushfires, nearby cyclone tracks.
   - **Infrastructure sparsity**: distance to the nearest Optus site, Optus sites within 5 km and within 10 km.
3. Each part is converted to a WA percentile, so the agreed **50 / 50 weighting** gives both parts equal influence.
4. The combined score is ranked: top 20% = **High**, next 30% = **Medium**, remaining 50% = **Low**.

**What it is not**

- It is **not trained** and has no target variable. The weights are set up front, so no train/test split is needed.
- The bands are **relative rankings**, not outage probabilities or predictions of when an outage will happen.
- Population is **not** part of the score. It is carried through for impact and prioritisation.

The Optus significant-outage register is used only as an **independent check** (section 7), never to tune the weights.

Requirements: `pandas` and `numpy`. `matplotlib` is optional (map in section 8). Runs locally and on LinuxONE.

## 1. Setup and configuration

In [ ]:
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)


def find_repo_root(start=None):
    """Walk up from the working directory until data/processed is found."""
    here = Path(start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data" / "processed").is_dir():
            return candidate
    raise FileNotFoundError("Could not find data/processed above " + str(here))


ROOT = find_repo_root()
PROCESSED = ROOT / "data" / "processed"
print("Repository root:", ROOT)

In [ ]:
# Agreed weighting: hazard and infrastructure sparsity count equally.
W_HAZARD = 0.5
W_INFRA = 1 - W_HAZARD

# Band cut-offs by rank: top 20% High, next 30% Medium, remaining 50% Low.
HIGH_SHARE = 0.20
MEDIUM_SHARE = 0.30

# Weightings used for the sensitivity check (hazard share).
SENSITIVITY_HAZARD_WEIGHTS = [0.4, 0.6]

# Scaling methods:
#   pct      - a 0-100 percentage divided by 100
#   log_high - log1p(value) / log1p(WA max); a higher value is worse
#   log_low  - 1 - log1p(value) / log1p(WA max); a lower value is worse
# Counts and distances are log-scaled because they are very skewed
# (e.g. 242 sites within 10 km in central Perth versus 0 in most of the outback).
FEATURES = {
    "hazard": [
        {"column": "bushfire_prone_area_pct_snapshot", "method": "pct",
         "label": "Bushfire-prone area", "template": "{:.0f}% of the locality is designated bushfire-prone"},
        {"column": "flood_any_mapped_extent_pct_snapshot", "method": "pct",
         "label": "Mapped floodplain", "template": "{:.0f}% of the locality is mapped floodplain"},
        {"column": "fire_per_year", "method": "log_high",
         "label": "Recorded bushfires", "template": "{:.1f} recorded bushfires per year"},
        {"column": "cyclone_tracks", "method": "log_high",
         "label": "Nearby cyclone tracks", "template": "{:.0f} cyclone(s) tracked within 100 km"},
    ],
    "infrastructure": [
        {"column": "nearest_optus_site_km", "method": "log_high",
         "label": "Distance to nearest Optus site", "template": "nearest Optus site is {:.1f} km away"},
        {"column": "optus_sites_within_5km", "method": "log_low", "group": "site_radius",
         "label": "Optus sites within 5 km", "template": "{:.0f} Optus site(s) within 5 km",
         "template_zero": "no Optus site within 5 km"},
        {"column": "optus_sites_within_10km", "method": "log_low", "group": "site_radius",
         "label": "Optus sites within 10 km", "template": "{:.0f} Optus site(s) within 10 km",
         "template_zero": "no Optus site within 10 km"},
    ],
}
FEATURE_INFO = {f["column"]: {**f, "part": part} for part, feats in FEATURES.items() for f in feats}

# The fire and cyclone history columns differ between the current planning
# table and the pre-outage-window modelling table, so they are mapped to
# generic names before scoring.
CURRENT_HISTORY_COLUMNS = {
    "dbca_recorded_bushfire_events_per_year": "fire_per_year",
    "nearby_cyclone_track_count": "cyclone_tracks",
}
PRE_WINDOW_HISTORY_COLUMNS = {
    "recorded_bushfires_per_year_before_window": "fire_per_year",
    "nearby_cyclone_track_count_before_window": "cyclone_tracks",
}

## 2. Load and validate the inputs

- `wa_locality_features.csv`: the current planning snapshot used to build the index. It has hazard snapshots, five-year fire and cyclone history to October 2026, and infrastructure.
- `wa_outage_modelling.csv`: used **only** for the register check. Its fire and cyclone history ends before the outage window.
- `wa_outages_clean.csv`: used to identify the 31 May – 1 Jun 2026 storm reports for the check.

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


INPUTS = {
    "features": PROCESSED / "wa_locality_features.csv",
    "modelling": PROCESSED / "wa_outage_modelling.csv",
    "outages": PROCESSED / "wa_outages_clean.csv",
}
features = pd.read_csv(INPUTS["features"], dtype={"sal_code_2021": str})
modelling = pd.read_csv(INPUTS["modelling"], dtype={"sal_code_2021": str})
outages = pd.read_csv(INPUTS["outages"], dtype={"event_id": str})

# Optional: compare against the checksums recorded by prepare_outage_data.py.
quality_path = PROCESSED / "outage_data_quality.json"
if quality_path.exists():
    recorded = json.loads(quality_path.read_text())["outputs"]
    for key, name in [("features", "wa_locality_features.csv"), ("modelling", "wa_outage_modelling.csv"),
                      ("outages", "wa_outages_clean.csv")]:
        status = "matches" if recorded[name]["sha256"] == sha256(INPUTS[key]) else "DIFFERS FROM"
        print(f"{name}: {status} the checksum in outage_data_quality.json")

print("features:", features.shape, " modelling:", modelling.shape, " outages:", outages.shape)

In [ ]:
raw_feature_columns = (
    ["bushfire_prone_area_pct_snapshot", "flood_any_mapped_extent_pct_snapshot",
     "nearest_optus_site_km", "optus_sites_within_5km", "optus_sites_within_10km"]
    + list(CURRENT_HISTORY_COLUMNS)
)
required = ["sal_code_2021", "locality", "longitude", "latitude", "population_2021", "flood_mapping_status"]

missing_cols = [c for c in required + raw_feature_columns if c not in features.columns]
assert not missing_cols, f"Missing columns: {missing_cols}"
assert features["sal_code_2021"].is_unique, "Duplicate locality codes"
assert features[raw_feature_columns].notna().all().all(), "Blank values in scoring features"
assert (features[raw_feature_columns] >= 0).all().all(), "Negative values in scoring features"
assert features[["bushfire_prone_area_pct_snapshot", "flood_any_mapped_extent_pct_snapshot"]].le(100).all().all()
assert set(modelling["sal_code_2021"]) == set(features["sal_code_2021"]), "Tables cover different localities"

# Stable row order makes the tie-breaks at the band cut-offs reproducible.
features = features.sort_values("sal_code_2021").reset_index(drop=True)
print("All input checks passed for", len(features), "localities.")
features[raw_feature_columns].describe().T

## 3. Scaling and scoring functions

Each feature becomes a 0–1 value where **1 = worst in WA**.

- Percentages are divided by 100.
- Counts and distances are log-scaled against the WA maximum.
- For "sites within" features the scale is reversed, so **0 sites = 1.0**.

The two parts are the mean of their scaled features. Each part is then converted to a WA percentile (0–1) before weighting. Without that step, infrastructure would dominate: its raw part scores spread about twice as widely as hazard's, so a nominal 50/50 split would not give equal influence.

In [ ]:
def scale_feature(values, method, ref_max):
    values = values.clip(lower=0)
    if method == "pct":
        return (values / 100).clip(0, 1)
    scaled = (np.log1p(values) / np.log1p(ref_max)).clip(0, 1) if ref_max > 0 else values * 0.0
    if method == "log_high":
        return scaled
    if method == "log_low":
        return 1 - scaled
    raise ValueError(method)


def build_index(df, w_hazard=W_HAZARD, ref_max=None):
    """Score every row. Returns (result DataFrame, reference maxima used for log scaling)."""
    if ref_max is None:
        ref_max = {col: float(df[col].max()) for col, info in FEATURE_INFO.items() if info["method"] != "pct"}
    out = pd.DataFrame(index=df.index)
    for part, feats in FEATURES.items():
        for f in feats:
            out["scaled_" + f["column"]] = scale_feature(df[f["column"]], f["method"], ref_max.get(f["column"], 0))
        out[part + "_score"] = out[["scaled_" + f["column"] for f in feats]].mean(axis=1)
        out[part + "_percentile"] = out[part + "_score"].rank(pct=True, method="average")
    out["susceptibility_score"] = (
        w_hazard * out["hazard_percentile"] + (1 - w_hazard) * out["infrastructure_percentile"]
    )
    return out, ref_max


def assign_bands(score, high_share=HIGH_SHARE, medium_share=MEDIUM_SHARE):
    """Rank-based bands. Ties at a cut-off are broken by row order (sorted locality code)."""
    rank = score.rank(ascending=False, method="first")
    n = len(score)
    band = np.where(rank <= high_share * n, "High",
                    np.where(rank <= (high_share + medium_share) * n, "Medium", "Low"))
    return pd.Series(band, index=score.index), rank.astype(int)

## 4. Build the index (current planning snapshot)

In [ ]:
planning = features.rename(columns=CURRENT_HISTORY_COLUMNS)
scores, REF_MAX = build_index(planning)
bands, ranks = assign_bands(scores["susceptibility_score"])

index = pd.concat([features[["sal_code_2021", "locality", "longitude", "latitude", "population_2021",
                             "flood_mapping_status"]],
                   planning[list(FEATURE_INFO)], scores], axis=1)
index["susceptibility_rank"] = ranks
index["susceptibility_band"] = bands
index["no_residents_2021"] = index["population_2021"] == 0

print("Log-scaling reference maxima:", REF_MAX)
print("Correlation between hazard and infrastructure parts:",
      round(index["hazard_percentile"].corr(index["infrastructure_percentile"]), 3))
index["susceptibility_band"].value_counts().reindex(["High", "Medium", "Low"])

In [ ]:
band_order = ["High", "Medium", "Low"]
summary = index.groupby("susceptibility_band").agg(
    localities=("sal_code_2021", "size"),
    residents_2021=("population_2021", "sum"),
    localities_without_residents=("no_residents_2021", "sum"),
    median_nearest_site_km=("nearest_optus_site_km", "median"),
    median_sites_within_10km=("optus_sites_within_10km", "median"),
    median_bushfire_prone_pct=("bushfire_prone_area_pct_snapshot", "median"),
    score_min=("susceptibility_score", "min"),
    score_max=("susceptibility_score", "max"),
).reindex(band_order)
summary.round(2)

Many of the highest-scoring localities are remote and have no residents, such as national parks and unincorporated land. They stay in the ranking because the score measures susceptibility, not impact. `no_residents_2021` lets the map and the resource optimisation set them aside.

In [ ]:
cols = ["locality", "population_2021", "susceptibility_score", "hazard_percentile",
        "infrastructure_percentile", "nearest_optus_site_km", "optus_sites_within_10km",
        "bushfire_prone_area_pct_snapshot"]
print("Top 15 High-band localities with at least 100 residents:")
display_top = index[(index["susceptibility_band"] == "High") & (index["population_2021"] >= 100)]
display_top.sort_values("susceptibility_score", ascending=False)[cols].head(15).round(3)

## 5. Explanations for each locality

For each locality we list up to three features whose scaled value is **above 0.5**, meaning they sit in the worse half of the scale. They are ranked by scaled value and shown with their measured values. The 5 km and 10 km site counts overlap, so only the stronger of the two is listed. These are the facts Granite will be given. They describe how the score was built; they are not causal claims.

In [ ]:
def describe_factors(row, max_factors=3, threshold=0.5):
    # The 5 km and 10 km site counts overlap, so only the stronger of the two is
    # listed (10 km wins a tie, e.g. when both are zero).
    best = {}
    for order, (col, info) in enumerate(reversed(list(FEATURE_INFO.items()))):
        scaled = row["scaled_" + col]
        if scaled <= threshold:
            continue
        template = info.get("template_zero") if row[col] == 0 and "template_zero" in info else info["template"]
        key = info.get("group", col)
        if key not in best or scaled > best[key][0]:
            best[key] = (scaled, order, template.format(row[col]))
    items = sorted(best.values(), key=lambda t: (-t[0], t[1]))
    if not items:
        return "no feature in the worse half of the WA scale"
    return "; ".join(text for _, _, text in items[:max_factors])


index["top_factors"] = index.apply(describe_factors, axis=1)
index.sort_values("susceptibility_rank")[["locality", "susceptibility_band", "top_factors"]].head(8)

## 6. Sensitivity to the weighting

We re-run the index with hazard weighted at 40% and 60%. A locality is **band-stable** if it gets the same band under all three weightings. Changes mostly happen to localities near a band cut-off.

In [ ]:
sensitivity = {}
stable = pd.Series(True, index=index.index)
for w in SENSITIVITY_HAZARD_WEIGHTS:
    alt_scores, _ = build_index(planning, w_hazard=w, ref_max=REF_MAX)
    alt_bands, _ = assign_bands(alt_scores["susceptibility_score"])
    col = f"band_hazard_weight_{int(w * 100)}"
    index[col] = alt_bands
    changed = int((alt_bands != index["susceptibility_band"]).sum())
    sensitivity[col] = {"hazard_weight": w, "localities_changing_band": changed,
                        "share_changing_band": round(changed / len(index), 4)}
    stable &= alt_bands == index["susceptibility_band"]
    print(f"Hazard weight {w:.0%}: {changed} localities ({changed / len(index):.1%}) change band")
    print(pd.crosstab(index["susceptibility_band"], alt_bands, rownames=["50/50"], colnames=[col])
          .reindex(index=band_order, columns=band_order), "\n")

index["band_stable"] = stable
sensitivity["band_stable_localities"] = int(stable.sum())
print("Band-stable localities:", int(stable.sum()), "of", len(index))

## 7. Independent check against the outage register

The register lists **reported significant outages**. Absence from it does not mean no outage, so this check **cannot measure accuracy**. It shows where the index and the reporting pattern agree or disagree.

To keep the check fair, the index is rebuilt with fire and cyclone history that **ends before the outage window** (from `wa_outage_modelling.csv`). The bushfire and flood snapshots are current designations. Nothing is fitted here and the weights are not changed based on the result.

"Rank agreement" is the AUC: the probability that a randomly chosen reported locality ranks above a randomly chosen unreported one. 0.5 means no agreement, below 0.5 means reported localities tend to rank *lower*.

In [ ]:
check_df = modelling.merge(
    features[["sal_code_2021", "bushfire_prone_area_pct_snapshot", "flood_any_mapped_extent_pct_snapshot"]],
    on="sal_code_2021", how="left",
).sort_values("sal_code_2021").reset_index(drop=True).rename(columns=PRE_WINDOW_HISTORY_COLUMNS)
check_scores, _ = build_index(check_df)
check_df["check_score"] = check_scores["susceptibility_score"]
check_df["check_band"], _ = assign_bands(check_df["check_score"])
check_df["reported"] = check_df["reported_outage_label"].fillna(0).astype(int)

# Storm cluster: weather-related WA-only reports starting 31 May - 1 Jun 2026 (WA time).
storm_events = set(outages.loc[
    (outages["weather_related_diagnostic_only"] == 1)
    & (outages["geographic_scope"] == "wa_only")
    & (outages["start_date_wa"].isin(["2026-05-31", "2026-06-01"])), "event_id"])
event_lists = check_df["source_event_ids_diagnostic_only"].fillna("").str.split(";")
check_df["storm_only_positive"] = (check_df["reported"] == 1) & event_lists.apply(
    lambda ids: all(i.strip() in storm_events for i in ids if i.strip()))
print(f"{len(storm_events)} storm reports; {int(check_df['storm_only_positive'].sum())} of "
      f"{int(check_df['reported'].sum())} reported localities come only from that storm")


def rank_agreement(score, label):
    label = np.asarray(label)
    n_pos, n_neg = label.sum(), len(label) - label.sum()
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    ranks = pd.Series(score).rank().to_numpy()
    return float((ranks[label == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))

In [ ]:
validation = {"overall": {"reported_localities": int(check_df["reported"].sum()),
                          "rank_agreement_auc": round(rank_agreement(check_df["check_score"], check_df["reported"]), 3)}}

# Reported localities skew heavily towards populated places, so compare within population thirds.
check_df["population_third"] = pd.qcut(check_df["population_2021"].rank(method="first"), 3,
                                       labels=["least populated", "middle", "most populated"])
for group, sub in check_df.groupby("population_third", observed=True):
    validation[str(group)] = {"reported_localities": int(sub["reported"].sum()),
                              "rank_agreement_auc": round(rank_agreement(sub["check_score"], sub["reported"]), 3)}

unreported = check_df["reported"] == 0
for name, mask in [("storm_only_reports", check_df["storm_only_positive"]),
                   ("other_reports", (check_df["reported"] == 1) & ~check_df["storm_only_positive"])]:
    sub = check_df[mask | unreported]
    validation[name] = {"reported_localities": int(mask.sum()),
                        "rank_agreement_auc": round(rank_agreement(sub["check_score"], sub["reported"]), 3)}

print(pd.DataFrame(validation).T)
print("\nBand share among reported vs unreported localities:")
pd.crosstab(check_df["check_band"], check_df["reported"].map({1: "reported", 0: "not reported"}),
            normalize="columns", rownames=["band"], colnames=["register"]).reindex(band_order).round(3)

**How to read this check**

- **Overall**, reported localities tend to rank *below* the rest. The register records "significant" outages, which need many affected services, so it concentrates in towns and suburbs with many customers and dense infrastructure. Those are the places the index ranks lowest on sparsity.
- **Within population thirds**, comparisons are fairer because reporting likelihood is more similar. Agreement is best among the most-populated localities, where most reports are.
- **This does not validate or invalidate the index.** The register cannot see small outages in sparse areas, which is exactly the gap the index is designed to highlight. Present this as a stated limitation, not an accuracy score.

## 8. Map of bands (optional, needs matplotlib)

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib not installed; skipping map")

if plt is not None:
    colours = {"Low": "#4c9f70", "Medium": "#e0a526", "High": "#c8402f"}
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    for ax, (title, box) in zip(axes, [("Western Australia", None),
                                       ("Perth and South West", (114.9, 117.0, -35.2, -31.2))]):
        for band in ["Low", "Medium", "High"]:
            sub = index[index["susceptibility_band"] == band]
            ax.scatter(sub["longitude"], sub["latitude"], s=8, c=colours[band], label=band, alpha=0.8)
        if box:
            ax.set_xlim(box[0], box[1])
            ax.set_ylim(box[2], box[3])
        ax.set_title(title)
        ax.set_aspect("equal")
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
    axes[0].legend(title="Relative susceptibility")
    fig.suptitle("Locality representative points by band (not coverage areas)")
    plt.tight_layout()
    plt.show()

## 9. Granite context

`granite_context()` packages one locality's band, score parts and measured factors as JSON. It is meant to be the **only** evidence passed to Granite, and the instructions tell Granite to keep the band and numbers unchanged. The Granite call itself comes once the LinuxONE runtime has been checked.

In [ ]:
CAVEATS = [
    "The band is a relative ranking among WA localities, not an outage probability or forecast.",
    "Infrastructure is the January 2025 ACCC Optus site snapshot; site identifiers are not necessarily unique towers.",
    "Distances are measured from one representative point inside the locality, not from every address.",
    "Flood values of 0 can mean the area is unmapped, not flood-free.",
    "Recorded bushfire history is incomplete statewide; 0 does not prove there were no fires.",
]


def granite_context(locality_or_code):
    row = index[(index["sal_code_2021"] == locality_or_code) | (index["locality"] == locality_or_code)]
    if row.empty:
        raise KeyError(locality_or_code)
    row = row.iloc[0]
    factors = [{"feature": info["label"], "part": info["part"], "measured_value": round(float(row[col]), 3),
                "scaled_0_to_1_worst": round(float(row["scaled_" + col]), 3)}
               for col, info in FEATURE_INFO.items()]
    return {
        "locality": row["locality"],
        "sal_code_2021": row["sal_code_2021"],
        "population_2021": int(row["population_2021"]),
        "band": row["susceptibility_band"],
        "rank": f"{int(row['susceptibility_rank'])} of {len(index)} (1 = most susceptible)",
        "hazard_percentile": round(float(row["hazard_percentile"]), 3),
        "infrastructure_sparsity_percentile": round(float(row["infrastructure_percentile"]), 3),
        "weights": {"hazard": W_HAZARD, "infrastructure_sparsity": W_INFRA},
        "band_stable_under_40_60_weighting": bool(row["band_stable"]),
        "main_factors": row["top_factors"],
        "all_factors": factors,
        "caveats": CAVEATS,
    }


GRANITE_INSTRUCTIONS = (
    "You explain a relative mobile-outage susceptibility band for a Western Australian locality. "
    "Use only the JSON evidence provided. Do not change the band, rank or any number. "
    "Do not describe the band as a probability or prediction. Mention the main factors and at least one caveat."
)

example = (index[(index["susceptibility_band"] == "High") & (index["population_2021"] >= 500)]
           .sort_values("susceptibility_rank").iloc[0]["locality"])
print(json.dumps(granite_context(example), indent=2))

## 10. Export

- `wa_susceptibility_index.csv`: one row per locality with bands, score parts, scaled features, explanations and sensitivity bands.
- `wa_susceptibility_index.metadata.json`: weights, cut-offs, scaling maxima, input checksums, sensitivity and check results. The disaster simulation should reuse the same `log_scaling_reference_max` values so that switching sites off changes scores on the same scale.

In [ ]:
OUTPUT_CSV = PROCESSED / "wa_susceptibility_index.csv"
OUTPUT_META = PROCESSED / "wa_susceptibility_index.metadata.json"

export_cols = (
    ["sal_code_2021", "locality", "longitude", "latitude", "population_2021", "no_residents_2021",
     "susceptibility_band", "susceptibility_rank", "susceptibility_score",
     "hazard_score", "hazard_percentile", "infrastructure_score", "infrastructure_percentile"]
    + list(FEATURE_INFO) + ["scaled_" + c for c in FEATURE_INFO]
    + ["flood_mapping_status", "top_factors"]
    + [c for c in index.columns if c.startswith("band_hazard_weight_")] + ["band_stable"]
)
export = index.sort_values("susceptibility_rank")[export_cols].rename(columns={
    "fire_per_year": "recorded_bushfires_per_year", "scaled_fire_per_year": "scaled_recorded_bushfires_per_year",
    "cyclone_tracks": "nearby_cyclone_track_count", "scaled_cyclone_tracks": "scaled_nearby_cyclone_track_count",
})
export.to_csv(OUTPUT_CSV, index=False, float_format="%.6f", lineterminator="\n")

metadata = {
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "rows": len(export),
    "csv_sha256": sha256(OUTPUT_CSV),
    "method": "Untrained transparent index: mean of scaled features per part, parts converted to WA percentiles, "
              "weighted sum, rank-based bands.",
    "weights": {"hazard": W_HAZARD, "infrastructure_sparsity": W_INFRA},
    "band_cutoffs": {"High": f"top {HIGH_SHARE:.0%}", "Medium": f"next {MEDIUM_SHARE:.0%}",
                     "Low": f"remaining {1 - HIGH_SHARE - MEDIUM_SHARE:.0%}",
                     "tie_break": "row order by sal_code_2021"},
    "features": {part: [{"column": f["column"], "method": f["method"], "label": f["label"]} for f in feats]
                 for part, feats in FEATURES.items()},
    "history_column_mapping": CURRENT_HISTORY_COLUMNS,
    "log_scaling_reference_max": REF_MAX,
    "inputs": {p.name: sha256(p) for p in INPUTS.values()},
    "sensitivity": sensitivity,
    "register_check": {
        "note": "Reported positives vs unreported (unknown) localities, using pre-outage-window fire/cyclone "
                "history. Not an accuracy measure: absence of a report is not confirmed normal service.",
        "storm_events": sorted(storm_events),
        "results": validation,
    },
    "limitations": CAVEATS + [
        "Population is excluded from the score; use it separately for impact and prioritisation.",
        "Bushfire-prone and floodplain values are current designation snapshots.",
        "No rainfall, local wind or live disaster conditions are included.",
        "Weights were set by the team, not fitted; the register check was not used to tune them.",
    ],
}
OUTPUT_META.write_text(json.dumps(metadata, indent=2) + "\n")
print(f"Wrote {OUTPUT_CSV.name} ({len(export)} rows) and {OUTPUT_META.name}")